# 🩺 DermaVision — Colab GPU Training

Train the EfficientNet-B3 skin lesion classifier on Google Colab GPU.

**Workflow:**
1. Verify GPU & clone repo
2. Download HAM10000 dataset from Kaggle
3. Verify data
4. Run training (two-phase: frozen backbone → fine-tune)
5. Download trained model checkpoint

> ⚠️ **Before starting:** Go to `Runtime → Change runtime type → GPU (T4)`

---
## Section 1 — Environment Setup
Clone the repository, install dependencies, and verify GPU.

In [ ]:
#@title 1.1 — Verify GPU
import torch

print("=" * 60)
print("🖥️  GPU CHECK")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  ✅ GPU available: {gpu_name}")
    print(f"  ✅ GPU memory:    {gpu_mem:.1f} GB")
    print(f"  ✅ CUDA version:  {torch.version.cuda}")
    print(f"  ✅ PyTorch:       {torch.__version__}")
else:
    print("  ❌ No GPU detected!")
    print("  → Go to Runtime → Change runtime type → GPU")
    raise RuntimeError("GPU is required for training.")

print("=" * 60)
!nvidia-smi

In [ ]:
#@title 1.2 — Clone Repository & Install Dependencies
import os

REPO_URL = "https://github.com/TryingtobeingNikhil/dermavision.git"
REPO_DIR = "/content/dermavision"

# Clone (or pull if already cloned)
if os.path.exists(REPO_DIR):
    print("📁 Repository already cloned. Pulling latest changes...")
    !cd {REPO_DIR} && git pull
else:
    print("📥 Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR}

# Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q -r {REPO_DIR}/requirements.txt

# Also ensure kaggle is installed
!pip install -q kaggle

print("\n✅ Environment setup complete!")

---
## Section 2 — Kaggle Dataset Download
Upload your `kaggle.json` credentials and download the HAM10000 dataset.

In [ ]:
#@title 2.1 — Upload Kaggle Credentials
import os
from google.colab import files

# Check if kaggle.json already exists
kaggle_dir = os.path.expanduser("~/.kaggle")
kaggle_json = os.path.join(kaggle_dir, "kaggle.json")

if os.path.exists(kaggle_json):
    print("✅ kaggle.json already configured.")
else:
    print("📤 Please upload your kaggle.json file...")
    print("   (Get it from https://www.kaggle.com/settings → API → Create New Token)\n")
    uploaded = files.upload()

    # Move to correct location
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(kaggle_json, "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod(kaggle_json, 0o600)
    print("\n✅ Kaggle credentials configured!")

In [ ]:
#@title 2.2 — Download HAM10000 Dataset
import os
import sys
import shutil

REPO_DIR = "/content/dermavision"

# Fix: download_data.py looks for 'kaggle' next to sys.executable
# (e.g. /usr/bin/kaggle), but on Colab it may be in /usr/local/bin/.
# Create a symlink so the existing script can find it.
kaggle_actual = shutil.which('kaggle')
python_dir = os.path.dirname(sys.executable)
kaggle_expected = os.path.join(python_dir, 'kaggle')

if kaggle_actual and not os.path.exists(kaggle_expected):
    !sudo ln -sf {kaggle_actual} {kaggle_expected}
    print(f"🔗 Symlinked {kaggle_actual} → {kaggle_expected}")

# Run the existing download script
print("\n📦 Downloading HAM10000 dataset...")
print("   This may take 5-10 minutes.\n")

!cd {REPO_DIR} && python scripts/download_data.py

print("\n✅ Dataset download complete!")

---
## Section 3 — Data Verification
Verify the dataset is correctly placed and inspect its structure.

In [ ]:
#@title 3.1 — Verify Dataset Structure
import os
import pandas as pd
from pathlib import Path

REPO_DIR = "/content/dermavision"
data_dir = Path(REPO_DIR) / "data"
processed_dir = data_dir / "processed"
metadata_path = data_dir / "metadata.csv"

print("=" * 60)
print("📊 DATASET VERIFICATION")
print("=" * 60)

# Check metadata
if metadata_path.exists():
    df = pd.read_csv(metadata_path)
    print(f"\n✅ metadata.csv found: {len(df)} total samples")
    print(f"   Columns: {list(df.columns)}")
    print(f"\n📋 Split distribution:")
    print(df['split'].value_counts().to_string())
    print(f"\n📋 Class distribution:")
    print(df['dx'].value_counts().to_string())
else:
    print("❌ metadata.csv NOT found! Run Section 2 first.")

# Check image folders
print(f"\n📂 Image folders:")
for part in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    part_dir = processed_dir / part
    if part_dir.exists():
        count = len(list(part_dir.glob("*.jpg")))
        print(f"   ✅ {part}: {count} images")
    else:
        print(f"   ❌ {part}: NOT found")

print("=" * 60)

---
## Section 4 — Training
Run the two-phase training pipeline:
- **Phase 1:** Train classifier head only (5 epochs, frozen backbone)
- **Phase 2:** Fine-tune full model (20 epochs, unfrozen backbone)

Training uses: EfficientNet-B3, Focal Loss, Weighted Sampling, CosineAnnealing LR.

In [ ]:
#@title 4.1 — Run Training
REPO_DIR = "/content/dermavision"

# Make the training script executable
!chmod +x {REPO_DIR}/scripts/train_colab.sh

# Launch training via the Colab helper script
!cd {REPO_DIR} && bash scripts/train_colab.sh

---
## Section 5 — Save & Download Model
Download the trained model checkpoint to your local machine.

In [ ]:
#@title 5.1 — List Saved Checkpoints
import os
from pathlib import Path

REPO_DIR = "/content/dermavision"
models_dir = Path(REPO_DIR) / "models"

print("=" * 60)
print("💾 SAVED CHECKPOINTS")
print("=" * 60)

if models_dir.exists():
    checkpoints = sorted(models_dir.glob("*.pth"))
    if checkpoints:
        for ckpt in checkpoints:
            size_mb = ckpt.stat().st_size / 1e6
            marker = "  ⭐" if "best" in ckpt.name else ""
            print(f"  {ckpt.name:35s}  {size_mb:7.1f} MB{marker}")
    else:
        print("  ❌ No checkpoints found. Run training first.")
else:
    print("  ❌ models/ directory not found.")

print("=" * 60)

In [ ]:
#@title 5.2 — Download Best Model
from google.colab import files
from pathlib import Path

REPO_DIR = "/content/dermavision"
best_model_path = Path(REPO_DIR) / "models" / "best_model.pth"
history_path = Path(REPO_DIR) / "logs" / "training_history.json"

# Download best model
if best_model_path.exists():
    size_mb = best_model_path.stat().st_size / 1e6
    print(f"📥 Downloading best_model.pth ({size_mb:.1f} MB)...")
    files.download(str(best_model_path))
else:
    print("❌ best_model.pth not found. Training may not have completed.")

# Optionally download training history
if history_path.exists():
    print(f"📥 Downloading training_history.json...")
    files.download(str(history_path))
else:
    print("ℹ️  training_history.json not found (optional).")

print("\n🎉 Done! Place best_model.pth in your local project's models/ directory.")